In [ ]:
# Cell 1 - Clone repo into /content/VDT_GD2
%cd /content
!if [ ! -d VDT_GD2 ]; then git clone https://github.com/Darkin72/VDT_GD2.git; fi
%cd /content/VDT_GD2
!git pull


In [ ]:
# Cell 2 - Hyperparameter configuration
EPOCHS = 50
BATCH_SIZE = 1
IMAGE_SIZE = 128
DEVICE = "cpu"       # change to "cuda" on a Colab GPU
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
print({
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "size": IMAGE_SIZE,
    "device": DEVICE, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY
})


In [ ]:
# Cell 3 - Train I-HAZE, O-HAZY and SOTS
!pip -q install pillow matplotlib numpy
%cd /content/VDT_GD2

!python -m solution.wavelet_dehaze.train dataset/I-HAZE/train --val-data dataset/I-HAZE/val --device {DEVICE} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --size {IMAGE_SIZE} --out haze_wavelet_i_haze.pt --history history_i_haze.json

!python -m solution.wavelet_dehaze.train dataset/O-HAZY/train --val-data dataset/O-HAZY/val --device {DEVICE} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --size {IMAGE_SIZE} --out haze_wavelet_o_hazy.pt --history history_o_hazy.json

# SOTS must be uploaded to dataset/SOTS/train and dataset/SOTS/val.
!if [ -d dataset/SOTS/train ]; then python -m solution.wavelet_dehaze.train dataset/SOTS/train --val-data dataset/SOTS/val --device {DEVICE} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --size {IMAGE_SIZE} --out haze_wavelet_sots.pt --history history_sots.json; else echo 'SOTS not found; skipped.'; fi


In [ ]:
# Cell 4 - Preview training metrics
import json
from pathlib import Path
import matplotlib.pyplot as plt

histories = {}
for name, path in [("I-HAZE","history_i_haze.json"), ("O-HAZY","history_o_hazy.json"), ("SOTS","history_sots.json")]:
    if Path(path).exists():
        histories[name] = json.loads(Path(path).read_text())
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for name, history in histories.items():
    e = [x["epoch"] for x in history]
    axes[0].plot(e, [x["train_loss"] for x in history], label=f"{name} train")
    axes[0].plot(e, [x.get("val_loss") for x in history], "--", label=f"{name} val")
    axes[1].plot(e, [x["train_psnr"] for x in history], label=f"{name} train")
    axes[1].plot(e, [x.get("val_psnr") for x in history], "--", label=f"{name} val")
    axes[2].plot(e, [x["train_ssim"] for x in history], label=f"{name} train")
    axes[2].plot(e, [x.get("val_ssim") for x in history], "--", label=f"{name} val")
for ax, title, ylabel in zip(axes, ["Loss", "PSNR", "SSIM"], ["loss", "dB", "score"]):
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.grid(alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
